# RAG Evaluation with RAGAS + Ollama

## Overview

After building a RAG pipeline, how do we know if it's actually working well? We need **evaluation metrics**.

[RAGAS](https://docs.ragas.io/) (Retrieval Augmented Generation Assessment) is a framework that evaluates RAG systems using an LLM as the judge. It scores three key aspects:

| Metric | What it measures | Score range |
|---|---|---|
| **Answer Correctness** | Is the answer factually correct compared to the ground truth? | 0 to 1 |
| **Faithfulness** | Is the answer grounded in the retrieved context (no hallucination)? | 0 to 1 |
| **Context Precision** | Are the retrieved chunks actually relevant and well-ranked? | 0 to 1 |

## Models Used

- **LLM judge**: `gemma3:4b` via Ollama (local)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)

> **Note:** We use Ollama instead of OpenAI so everything runs locally with no API key needed.

---
## Step 0: Import Packages

In [1]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama, OllamaEmbeddings
from datasets import Dataset

/tmp/ipykernel_3314164/1566550172.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_correctness, context_precision
/tmp/ipykernel_3314164/1566550172.py:2: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import faithfulness, answer_correctness, context_precision
/tmp/ipykernel_3314164/1566550172.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, answer_corr

---
## Step 1: Set Up the LLM Judge and Embedding Model

RAGAS uses an LLM to *judge* the quality of RAG outputs. It also uses an embedding model for semantic similarity in certain metrics.

We wrap our Ollama models with RAGAS-compatible wrappers.

In [2]:
ollama_llm = LangchainLLMWrapper(ChatOllama(model="gemma3:4b"))
embedding_model = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="mxbai-embed-large:335m"))

print("LLM judge and embedding model ready")

LLM judge and embedding model ready


/tmp/ipykernel_3314164/4293079922.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ollama_llm = LangchainLLMWrapper(ChatOllama(model="gemma3:4b"))
/tmp/ipykernel_3314164/4293079922.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embedding_model = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="mxbai-embed-large:335m"))


---
## Step 2: Test Answer Correctness

**Answer Correctness** measures whether the predicted answer matches the ground truth.

It combines:
- **Factual similarity** — does it say the same facts? (checked by the LLM)
- **Semantic similarity** — is the meaning close? (checked by embeddings)

Here we test: the ground truth is `"Madrid is the capital of Spain."` and the predicted answer is just `"MadriD."` — a correct but very brief answer.

In [3]:
eval_dataset = Dataset.from_dict({
    "question":     ["What is the capital of Spain?"],
    "answer":       ["MadriD."],
    "ground_truth": ["Madrid is the capital of Spain."],
    "contexts":     [["The capital of Spain is Madrid."]],
})

result = evaluate(
    dataset=eval_dataset,
    metrics=[answer_correctness],
    llm=ollama_llm,
    embeddings=embedding_model,
)

print(result)
print(result.to_pandas())

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'answer_correctness': 0.9717}
                      user_input                 retrieved_contexts response  \
0  What is the capital of Spain?  [The capital of Spain is Madrid.]  MadriD.   

                         reference  answer_correctness  
0  Madrid is the capital of Spain.            0.971657  


The score is close to 1.0 because `"MadriD"` is factually correct, even though it's not a complete sentence.

---
## Step 3: Test Faithfulness

**Faithfulness** measures whether the answer is *grounded in the retrieved context* — i.e., no hallucination.

RAGAS does this by:
1. Breaking the answer into individual factual claims.
2. Checking if each claim can be inferred from the context.

$$\text{faithfulness} = \frac{\text{claims supported by context}}{\text{total claims}}$$

Here the answer is `"6"` and the context says `"3+3=6"` — perfectly grounded, so we expect a score of 1.0.

In [4]:
eval_dataset = Dataset.from_dict({
    "question":     ["what is 3+3?"],
    "answer":       ["6"],
    "ground_truth": ["6"],
    "contexts":     [["3+3=6"]],
})

result = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness],
    llm=ollama_llm,
    embeddings=embedding_model,
)

print(result)
print(result.to_pandas())

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'faithfulness': 1.0000}
     user_input retrieved_contexts response reference  faithfulness
0  what is 3+3?            [3+3=6]        6         6           1.0


---
## Step 4: Test Context Precision

**Context Precision** measures whether the retrieved chunks are relevant and well-ranked.

If the truly relevant chunk is ranked first, precision is high. If irrelevant chunks are ranked above it, precision drops.

Here we simulate a retrieval that returned 3 chunks:
1. `"this is a test context"` — irrelevant
2. `"mike is a cat"` — irrelevant
3. `"if the shoes don't fit, then go somewhere else."` — relevant

The relevant chunk is ranked last (position 3), so precision should be low (~0.33).

In [5]:
eval_dataset = Dataset.from_dict({
    "question":     ["What if these shoes don't fit?"],
    "answer":       ["if the shoes don't fit, then go somewhere else."],
    "ground_truth": ["then go somewhere else."],
    "contexts":     [[
        "this is a test context",
        "mike is a cat",
        "if the shoes don't fit, then go somewhere else."
    ]],
})

result = evaluate(
    dataset=eval_dataset,
    metrics=[context_precision],
    llm=ollama_llm,
    embeddings=embedding_model,
)

print(result)
print(result.to_pandas())

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'context_precision': 0.3333}
                       user_input  \
0  What if these shoes don't fit?   

                                  retrieved_contexts  \
0  [this is a test context, mike is a cat, if the...   

                                          response                reference  \
0  if the shoes don't fit, then go somewhere else.  then go somewhere else.   

   context_precision  
0           0.333333  


As expected, context precision is ~0.33 because only 1 out of 3 retrieved chunks is relevant, and it's ranked last.

---
## Step 5: Evaluate Multiple Questions with All Metrics at Once

In practice, you evaluate your RAG system across **many questions** and **multiple metrics** simultaneously.

Here we test two questions at once and measure all three metrics.

> **Important:** The `"contexts"` field must be a **list of lists** — each question gets its own list of retrieved chunks.

In [6]:
eval_dataset = Dataset.from_dict({
    "question":     ["What is the capital of Spain?", "What is 3+3"],
    "answer":       ["Madrid is the capital of Spain.", "6"],
    "ground_truth": ["MadriD.", "6"],
    "contexts":     [
        ["The capital of Spain is Madrid."],   # contexts for question 1
        ["3+3=6"],                              # contexts for question 2
    ],
})

result = evaluate(
    dataset=eval_dataset,
    metrics=[answer_correctness, faithfulness, context_precision],
    llm=ollama_llm,
    embeddings=embedding_model,
)

print("\nAggregate scores:")
print(result)

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


Aggregate scores:
{'answer_correctness': 0.9858, 'faithfulness': 1.0000, 'context_precision': 1.0000}


---
## Step 6: View Per-Question Results

The aggregate scores above are averages. Let's see the per-question breakdown.

In [7]:
df = result.to_pandas()
print(df.to_string(index=False))

                   user_input                retrieved_contexts                        response reference  answer_correctness  faithfulness  context_precision
What is the capital of Spain? [The capital of Spain is Madrid.] Madrid is the capital of Spain.   MadriD.            0.971657           1.0                1.0
                  What is 3+3                           [3+3=6]                               6         6            1.000000           1.0                1.0


---
## Summary

| What we tested | Metric | Key takeaway |
|---|---|---|
| Abbreviated but correct answer | Answer Correctness | Even `"MadriD."` scores high against the full ground truth |
| Answer fully grounded in context | Faithfulness | Score = 1.0 when every claim comes from the context |
| Irrelevant chunks ranked above relevant ones | Context Precision | Score drops when the retriever ranks poorly |
| Multiple questions + all metrics | All three | RAGAS evaluates everything in one call |